In [1]:
from qdrant_client import QdrantClient
import tqdm as notebook_tqdm

# Connect to your Qdrant instance
# qc = QdrantClient(path="persistent_db/vector_db/qdrant")  # Use the same path as in your code

/Users/deenuy/.virtualenvs/agentic-ai-workspace/chatbot_openAI/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def get_qdrant_client():
    """Returns a Qdrant client that works in both development and production."""
    try:
        # First try URL connection
        client = QdrantClient(url="http://localhost:6333")
        # Test connection
        client.get_collections()
        print("Using HTTP connection to Qdrant")
        return client
    except:
        # Fall back to direct file access
        client = QdrantClient(path="persistent_db/vector_db/qdrant")
        print("Using direct file access to Qdrant")
        return client

In [5]:
get_qdrant_client()

Using HTTP connection to Qdrant


In [2]:
# Get both clients
http_client = QdrantClient(url="http://localhost:6333")
file_client = QdrantClient(path="persistent_db/vector_db/qdrant")

# Check collections with both
http_collections = http_client.get_collections()
file_collections = file_client.get_collections()

print("HTTP collections:", [c.name for c in http_collections.collections])
print("File collections:", [c.name for c in file_collections.collections])

# Try to create a test collection via HTTP
try:
    from qdrant_client.http import models as rest

    http_client.recreate_collection(
        collection_name="test-collection",
        vectors_config={
            "text": rest.VectorParams(
                size=1536,
                distance=rest.Distance.COSINE,
            )
        }
    )
    print("Test collection created successfully via HTTP")

    # Check collections again
    http_collections = http_client.get_collections()
    file_collections = file_client.get_collections()

    print("HTTP collections after test:", [c.name for c in http_collections.collections])
    print("File collections after test:", [c.name for c in file_collections.collections])
except Exception as e:
    print(f"Error creating test collection: {e}")

HTTP collections: []
File collections: ['llamaindex-docs']


/var/folders/2_/v9nzhgd118s4dqpsthz07ltc0000gp/T/ipykernel_81546/453255856.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  http_client.recreate_collection(


Test collection created successfully via HTTP
HTTP collections after test: ['test-collection']
File collections after test: ['llamaindex-docs']


In [3]:
# 1. Export data from file-based client
file_client = QdrantClient(path="persistent_db/vector_db/qdrant")
points, _ = file_client.scroll(
    collection_name="llamaindex-docs",
    limit=10000,
    with_vectors=True,
    with_payload=True
)

# 2. Create collection via HTTP client
http_client = QdrantClient(url="http://localhost:6333")
from qdrant_client.http import models as rest

http_client.recreate_collection(
    collection_name="llamaindex-docs",
    vectors_config={
        "text": rest.VectorParams(
            size=1536,
            distance=rest.Distance.COSINE,
        )
    }
)

# 3. Import points to the new collection
http_client.upsert(
    collection_name="llamaindex-docs",
    points=[
        rest.PointStruct(
            id=p.id,
            vector=p.vector,
            payload=p.payload
        ) for p in points
    ]
)

RuntimeError: Storage folder persistent_db/vector_db/qdrant is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [2]:
# Get list of collections
collections = qc.get_collections()
print("Collections:", [c.name for c in collections.collections])

Collections: ['llamaindex-docs']


In [3]:
# Try URL connection to compare with path connection
qc_url = QdrantClient(url="http://localhost:6333")
collections_url = qc_url.get_collections()
print("URL Connection Collections:", [c.name for c in collections_url.collections])

URL Connection Collections: []


In [ ]:
# If you have a collection
if collections.collections:
    # Get collection info for the first collection
    collection_name = collections.collections[0].name
    info = qc.get_collection(collection_name)
    print(f"\nInfo for collection '{collection_name}':")
    print(f"  Vectors config: {info.config.params.vectors}")
    print(f"  Points count: {info.points_count}")

    # Get first 5 points
    points = qc.scroll(
        collection_name=collection_name,
        limit=5,
        with_vectors=False,  # Set to True if you want to see the vectors
        with_payload=True
    )

    print("\nSample points:")
    for point in points[0]:
        print(f"  ID: {point.id}")
        print(f"  Payload: {point.payload}")
        print("  ---")